###1) Konfiguracja zakresu CAISO

In [0]:
from datetime import (
    datetime,
    date,
    time,
    timezone,
    timedelta,
)
from pathlib import Path
from zoneinfo import ZoneInfo

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

dbutils.widgets.text(
    "evaluation_start_date_local",
    "2024-07-01"
)

dbutils.widgets.text(
    "evaluation_end_date_local_exclusive",
    "2024-11-01"
)

dbutils.widgets.text(
    "fmm_history_buffer_days",
    "1"
)

EVALUATION_START_DATE_TEXT = dbutils.widgets.get(
    "evaluation_start_date_local"
)

EVALUATION_END_DATE_TEXT = dbutils.widgets.get(
    "evaluation_end_date_local_exclusive"
)

FMM_HISTORY_BUFFER_DAYS = int(
    dbutils.widgets.get(
        "fmm_history_buffer_days"
    )
)

if FMM_HISTORY_BUFFER_DAYS < 1:
    raise ValueError(
        "Bufor historii FMM musi wynosić "
        "co najmniej jeden dzień."
    )

MARKET_TIMEZONE = "America/Los_Angeles"
LOCAL_TIMEZONE = ZoneInfo(MARKET_TIMEZONE)

CAISO_NODE = "TH_NP15_GEN-APND"

CAISO_ENDPOINT = (
    "https://oasis.caiso.com/"
    "oasisapi/SingleZip"
)

BASE_DIRECTORY = Path(
    "/Volumes/workspace/energy_bronze/"
    "landing/caiso/np15"
)

DAM_DIRECTORY = (
    BASE_DIRECTORY / "dam"
)

FMM_DIRECTORY = (
    BASE_DIRECTORY / "fmm"
)

DAM_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

FMM_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

evaluation_start_local = datetime.combine(
    date.fromisoformat(
        EVALUATION_START_DATE_TEXT
    ),
    time.min,
    tzinfo=LOCAL_TIMEZONE
)

evaluation_end_local = datetime.combine(
    date.fromisoformat(
        EVALUATION_END_DATE_TEXT
    ),
    time.min,
    tzinfo=LOCAL_TIMEZONE
)

if evaluation_end_local <= evaluation_start_local:
    raise ValueError(
        "Data końcowa musi być późniejsza "
        "od daty początkowej."
    )

EVALUATION_START_UTC = (
    evaluation_start_local
    .astimezone(timezone.utc)
)

EVALUATION_END_UTC = (
    evaluation_end_local
    .astimezone(timezone.utc)
)

DAM_START_UTC = EVALUATION_START_UTC
DAM_END_UTC = EVALUATION_END_UTC

FMM_START_UTC = (
    EVALUATION_START_UTC
    - timedelta(
        days=FMM_HISTORY_BUFFER_DAYS
    )
)

FMM_END_UTC = EVALUATION_END_UTC


def build_request_windows(
    start_utc,
    end_utc,
    window_size
):
    request_windows = []
    current_start_utc = start_utc

    while current_start_utc < end_utc:
        current_end_utc = min(
            current_start_utc + window_size,
            end_utc
        )

        request_windows.append((
            current_start_utc,
            current_end_utc
        ))

        current_start_utc = (
            current_end_utc
        )

    return request_windows


DAM_REQUEST_WINDOWS = build_request_windows(
    DAM_START_UTC,
    DAM_END_UTC,
    timedelta(days=1)
)

FMM_REQUEST_WINDOWS = build_request_windows(
    FMM_START_UTC,
    FMM_END_UTC,
    timedelta(hours=1)
)

EXPECTED_DAM_ARCHIVE_COUNT = len(
    DAM_REQUEST_WINDOWS
)

EXPECTED_FMM_ARCHIVE_COUNT = len(
    FMM_REQUEST_WINDOWS
)

EXPECTED_DAM_INTERVAL_COUNT = int(
    (
        DAM_END_UTC
        - DAM_START_UTC
    ).total_seconds()
    / 3600
)

EXPECTED_FMM_INTERVAL_COUNT = int(
    (
        FMM_END_UTC
        - FMM_START_UTC
    ).total_seconds()
    / (15 * 60)
)

EXPECTED_GOLD_TARGET_INTERVAL_COUNT = int(
    (
        EVALUATION_END_UTC
        - EVALUATION_START_UTC
    ).total_seconds()
    / (15 * 60)
)

print(
    "Zakres lokalny:",
    evaluation_start_local,
    "—",
    evaluation_end_local
)

print(
    "DAM UTC:",
    DAM_START_UTC,
    "—",
    DAM_END_UTC
)

print(
    "FMM UTC z buforem:",
    FMM_START_UTC,
    "—",
    FMM_END_UTC
)

print(
    "Archiwa DAM:",
    EXPECTED_DAM_ARCHIVE_COUNT
)

print(
    "Archiwa FMM:",
    EXPECTED_FMM_ARCHIVE_COUNT
)

print(
    "Interwały DAM:",
    EXPECTED_DAM_INTERVAL_COUNT
)

print(
    "Interwały FMM z buforem:",
    EXPECTED_FMM_INTERVAL_COUNT
)

print(
    "Docelowe interwały Gold:",
    EXPECTED_GOLD_TARGET_INTERVAL_COUNT
)

print("Katalog DAM:", DAM_DIRECTORY)
print("Katalog FMM:", FMM_DIRECTORY)

###2) Funkcje obsługi CAISO

In [0]:
from io import BytesIO
from zipfile import (
    ZipFile,
    BadZipFile,
)
import time as time_module

import requests

http_session = requests.Session()

http_session.headers.update({
    "User-Agent": (
        "energy-data-engineering-thesis/2.0"
    ),
    "Accept": (
        "application/zip,"
        "application/octet-stream,"
        "*/*"
    ),
})


def caiso_datetime(timestamp_utc):
    return timestamp_utc.strftime(
        "%Y%m%dT%H:%M-0000"
    )


def dam_target_path(
    start_utc,
    end_utc
):
    return (
        DAM_DIRECTORY
        / (
            "dam_"
            f"{start_utc.strftime('%Y%m%dT%H%MZ')}_"
            f"{end_utc.strftime('%Y%m%dT%H%MZ')}"
            ".zip"
        )
    )


def fmm_target_path(start_utc):
    return (
        FMM_DIRECTORY
        / (
            "fmm_"
            f"{start_utc.strftime('%Y%m%dT%H%MZ')}"
            ".zip"
        )
    )


def valid_caiso_zip(file_path):
    file_path = Path(file_path)

    if not file_path.exists():
        return False

    if file_path.stat().st_size == 0:
        return False

    try:
        with ZipFile(
            file_path,
            "r"
        ) as archive:
            csv_members = [
                file_info
                for file_info
                in archive.infolist()
                if (
                    not file_info.is_dir()
                    and file_info.filename
                        .lower()
                        .endswith(".csv")
                    and file_info.file_size > 0
                )
            ]

            if not csv_members:
                return False

            if archive.testzip() is not None:
                return False

        return True

    except (BadZipFile, OSError):
        return False


def download_caiso_zip(
    target_path,
    request_parameters,
    description,
    max_attempts=4
):
    target_path = Path(target_path)

    if valid_caiso_zip(target_path):
        return "skipped_existing", ""

    temporary_path = target_path.with_suffix(
        target_path.suffix + ".part"
    )

    temporary_path.unlink(
        missing_ok=True
    )

    last_error = ""

    for attempt in range(
        1,
        max_attempts + 1
    ):
        try:
            response = http_session.get(
                CAISO_ENDPOINT,
                params=request_parameters,
                timeout=(10, 180)
            )

            response.raise_for_status()

            response_bytes = response.content

            if not response_bytes:
                raise RuntimeError(
                    "CAISO zwróciło pustą odpowiedź."
                )

            with ZipFile(
                BytesIO(response_bytes),
                "r"
            ) as archive:
                csv_members = [
                    file_info
                    for file_info
                    in archive.infolist()
                    if (
                        not file_info.is_dir()
                        and file_info.filename
                            .lower()
                            .endswith(".csv")
                        and file_info.file_size > 0
                    )
                ]

                if not csv_members:
                    response_preview = []

                    for file_info in (
                        archive.infolist()[:3]
                    ):
                        if file_info.is_dir():
                            continue

                        response_preview.append(
                            archive.read(
                                file_info.filename
                            )[:500].decode(
                                "utf-8",
                                errors="replace"
                            )
                        )

                    raise RuntimeError(
                        "ZIP nie zawiera CSV. "
                        + " | ".join(
                            response_preview
                        )
                    )

                bad_member = archive.testzip()

                if bad_member is not None:
                    raise RuntimeError(
                        "Uszkodzony element ZIP: "
                        f"{bad_member}"
                    )

            temporary_path.write_bytes(
                response_bytes
            )

            if not valid_caiso_zip(
                temporary_path
            ):
                raise RuntimeError(
                    "Zapisany ZIP nie przeszedł "
                    "kontroli jakości."
                )

            temporary_path.replace(
                target_path
            )

            return "downloaded", ""

        except Exception as error:
            last_error = (
                f"{type(error).__name__}: "
                f"{str(error)}"
            )

            temporary_path.unlink(
                missing_ok=True
            )

            if attempt < max_attempts:
                wait_seconds = 2 ** attempt

                print(
                    f"{description}: "
                    f"próba {attempt}/"
                    f"{max_attempts} "
                    f"nieudana. Ponowienie za "
                    f"{wait_seconds} s. "
                    f"Błąd: {last_error}"
                )

                time_module.sleep(
                    wait_seconds
                )

    return "failed", last_error


print(
    "Funkcje pobierania CAISO są gotowe."
)

###3) Pobieranie DAM

In [0]:
dam_download_results = []

for request_number, (
    request_start_utc,
    request_end_utc
) in enumerate(
    DAM_REQUEST_WINDOWS,
    start=1
):
    target_path = dam_target_path(
        request_start_utc,
        request_end_utc
    )

    request_parameters = {
        "queryname": "PRC_LMP",
        "startdatetime": caiso_datetime(
            request_start_utc
        ),
        "enddatetime": caiso_datetime(
            request_end_utc
        ),
        "version": "1",
        "market_run_id": "DAM",
        "node": CAISO_NODE,
        "resultformat": "6",
    }

    description = (
        f"DAM {request_number}/"
        f"{EXPECTED_DAM_ARCHIVE_COUNT} "
        f"{request_start_utc.isoformat()}"
    )

    status, error_text = (
        download_caiso_zip(
            target_path=target_path,
            request_parameters=(
                request_parameters
            ),
            description=description
        )
    )

    dam_download_results.append({
        "start_utc": request_start_utc,
        "end_utc": request_end_utc,
        "target_path": str(target_path),
        "status": status,
        "error": error_text,
    })

    print(
        f"{description}: {status}"
    )

    if error_text:
        print(f"  Błąd: {error_text}")

    time_module.sleep(0.25)


expected_dam_paths = [
    dam_target_path(
        request_start_utc,
        request_end_utc
    )
    for (
        request_start_utc,
        request_end_utc
    ) in DAM_REQUEST_WINDOWS
]

valid_dam_paths = [
    file_path
    for file_path in expected_dam_paths
    if valid_caiso_zip(file_path)
]

failed_dam_results = [
    result
    for result in dam_download_results
    if result["status"] == "failed"
]

downloaded_dam_count = sum(
    result["status"] == "downloaded"
    for result in dam_download_results
)

skipped_dam_count = sum(
    result["status"]
    == "skipped_existing"
    for result in dam_download_results
)

print()
print("=" * 70)
print("PODSUMOWANIE DAM")
print("=" * 70)

print(
    "Pominięte istniejące:",
    skipped_dam_count
)

print(
    "Pobrane:",
    downloaded_dam_count
)

print(
    "Nieudane:",
    len(failed_dam_results)
)

print(
    "Poprawne archiwa:",
    f"{len(valid_dam_paths)}/"
    f"{EXPECTED_DAM_ARCHIVE_COUNT}"
)

if failed_dam_results:
    for result in failed_dam_results:
        print(
            result["start_utc"],
            result["error"]
        )

if (
    len(valid_dam_paths)
    != EXPECTED_DAM_ARCHIVE_COUNT
):
    raise RuntimeError(
        "Pobieranie DAM jest niekompletne. "
        "Uruchom ponownie komórkę 3."
    )

print(
    "Pobieranie DAM zakończone poprawnie."
)

###3A) Obsługa limitu HTTP 429

In [0]:
import random

base_download_caiso_zip = (
    download_caiso_zip
)


def download_caiso_zip(
    target_path,
    request_parameters,
    description,
    max_attempts=8
):
    last_error = ""

    for attempt in range(
        1,
        max_attempts + 1
    ):
        status, error_text = (
            base_download_caiso_zip(
                target_path=target_path,
                request_parameters=(
                    request_parameters
                ),
                description=description,
                max_attempts=1
            )
        )

        if status in {
            "downloaded",
            "skipped_existing",
        }:
            return status, ""

        last_error = error_text

        is_rate_limit_error = (
            "429" in error_text
            or "Too Many Requests"
                in error_text
        )

        if is_rate_limit_error:
            wait_seconds = min(
                60 * attempt,
                300
            )
        else:
            wait_seconds = min(
                2 ** attempt,
                60
            )

        wait_seconds += random.uniform(
            0.0,
            3.0
        )

        print(
            f"{description}: "
            f"próba {attempt}/"
            f"{max_attempts} nieudana. "
            f"Oczekiwanie "
            f"{wait_seconds:.1f} s. "
            f"Błąd: {error_text}"
        )

        if attempt < max_attempts:
            time_module.sleep(
                wait_seconds
            )

    return "failed", last_error


print(
    "Aktywna bezpieczna obsługa "
    "limitu CAISO 429."
)

###4) Pobieranie FMM z buforem

In [0]:
fmm_download_results = []

for request_number, (
    request_start_utc,
    request_end_utc
) in enumerate(
    FMM_REQUEST_WINDOWS,
    start=1
):
    target_path = fmm_target_path(
        request_start_utc
    )

    request_parameters = {
        "queryname": "PRC_RTPD_LMP",
        "startdatetime": caiso_datetime(
            request_start_utc
        ),
        "enddatetime": caiso_datetime(
            request_end_utc
        ),
        "version": "3",
        "market_run_id": "RTPD",
        "node": CAISO_NODE,
        "resultformat": "6",
    }

    description = (
        f"FMM {request_number}/"
        f"{EXPECTED_FMM_ARCHIVE_COUNT} "
        f"{request_start_utc.isoformat()}"
    )

    status, error_text = (
        download_caiso_zip(
            target_path=target_path,
            request_parameters=(
                request_parameters
            ),
            description=description
        )
    )

    fmm_download_results.append({
        "start_utc": request_start_utc,
        "end_utc": request_end_utc,
        "target_path": str(target_path),
        "status": status,
        "error": error_text,
    })

    if (
        request_number == 1
        or request_number % 25 == 0
        or request_number
            == EXPECTED_FMM_ARCHIVE_COUNT
        or status == "failed"
    ):
        print(
            f"{description}: {status}"
        )

        if error_text:
            print(
                f"  Błąd: {error_text}"
            )

    if status == "downloaded":
        time_module.sleep(6.5)


expected_fmm_paths = [
    fmm_target_path(
        request_start_utc
    )
    for (
        request_start_utc,
        request_end_utc
    ) in FMM_REQUEST_WINDOWS
]

valid_fmm_paths = [
    file_path
    for file_path in expected_fmm_paths
    if valid_caiso_zip(file_path)
]

failed_fmm_results = [
    result
    for result in fmm_download_results
    if result["status"] == "failed"
]

downloaded_fmm_count = sum(
    result["status"] == "downloaded"
    for result in fmm_download_results
)

skipped_fmm_count = sum(
    result["status"]
    == "skipped_existing"
    for result in fmm_download_results
)

print()
print("=" * 70)
print("PODSUMOWANIE FMM")
print("=" * 70)

print(
    "Pominięte istniejące:",
    skipped_fmm_count
)

print(
    "Pobrane:",
    downloaded_fmm_count
)

print(
    "Nieudane:",
    len(failed_fmm_results)
)

print(
    "Poprawne archiwa:",
    f"{len(valid_fmm_paths)}/"
    f"{EXPECTED_FMM_ARCHIVE_COUNT}"
)

if failed_fmm_results:
    print("\nNieudane przedziały:")

    for result in failed_fmm_results:
        print(
            result["start_utc"],
            result["error"]
        )

if (
    len(valid_fmm_paths)
    != EXPECTED_FMM_ARCHIVE_COUNT
):
    raise RuntimeError(
        "Pobieranie FMM jest niekompletne. "
        "Uruchom tę komórkę ponownie. "
        "Poprawne archiwa zostaną pominięte."
    )

print(
    "Pobieranie FMM zakończone poprawnie."
)